### Dependencias

Instala una sola vez las versiones fijadas en `requirements.txt` desde la raíz del proyecto: `pip install -r requirements.txt`.

In [1]:
# Dependencias declaradas en ../requirements.txt; no se desinstalan paquetes desde el notebook.


### imports

In [2]:
import os
import hashlib
import json
from pathlib import Path
import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore') 

print("listo")


listo


In [3]:
RUTA_PROYECTO = Path.cwd().resolve()
if not (RUTA_PROYECTO / "data").is_dir():
    RUTA_PROYECTO = RUTA_PROYECTO.parent

RUTA_CORPUS = RUTA_PROYECTO / "data" / "corpus_resenas.csv"

df = pd.read_csv(RUTA_CORPUS)
print(f"{len(df)} reseñas cargadas")
df[["texto", "tipo_lugar", "calificacion", "lugar"]].head(3)


5018 reseñas cargadas


,texto,tipo_lugar,calificacion,lugar
0,"Tour fabuloso, bellísimo el bosque nuboso de M...",parque,5.0,Excursión a los puentes colgantes de Monteverde
1,Vale la pena ir a ver los puentes colgantes de...,parque,4.0,Excursión a los puentes colgantes de Monteverde
2,"La actividad es muy bonita, sin embargo hay do...",parque,4.0,Excursión a los puentes colgantes de Monteverde


### chunking

un chunk es cada pedacito en el que partimos el texto antes de convertirlo en vector. si
metemos la reseña completa sin partir se pierde precision en la busqueda cuando el texto es
largo, pero tampoco conviene pasarse partiendo porque se pierde contexto.

como nuestras reseñas ya son cortas la diferencia no es tan grande como con un texto largo,
pero se nota en la cantidad de chunks que salen de cada una

In [4]:
def chunking_oraciones(texto, oraciones_por_chunk=3, overlap_oraciones=1):
    # separa por punto, signo de exclamacion o interrogacion
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    oraciones = [o.strip() for o in oraciones if o.strip()]

    chunks = []
    i = 0
    while i < len(oraciones):
        grupo = oraciones[i:i + oraciones_por_chunk]
        chunk = " ".join(grupo)
        if chunk:
            chunks.append(chunk)
        i += oraciones_por_chunk - overlap_oraciones  # el overlap es para no perder contexto entre chunks
    return chunks


def chunking_parrafos(texto, min_longitud=50):
    # separa por parrafos (doble salto de linea)
    parrafos = re.split(r'\n\s*\n', texto)
    parrafos = [p.strip() for p in parrafos if p.strip()]

    # si un parrafo queda muy corto lo pega con el siguiente para que no quede un chunk vacio de info
    chunks = []
    buffer = ""
    for p in parrafos:
        if len(buffer) + len(p) < min_longitud * 3:
            buffer += " " + p
        else:
            if buffer.strip():
                chunks.append(buffer.strip())
            buffer = p
    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks


In [5]:
# recorre el corpus reseña por reseña y le va aplicando el chunking, pegandole la metadata a cada pedazo
# (esto no lo vimos exactamente asi en clase porque alla era un solo documento, aca son miles de reseñas cortas)
def aplicar_chunking(df, funcion, **kwargs):
    resultado = []
    for idx, fila in df.iterrows():
        for fragmento in funcion(str(fila["texto"]), **kwargs):
            resultado.append({
                "texto": fragmento,
                "lugar": fila["lugar"],
                "tipo_lugar": fila["tipo_lugar"],
                "calificacion": fila["calificacion"],
                "polaridad": fila["polaridad"],
                "fuente": fila["fuente"],
            })
    return resultado


In [6]:
# comparamos las dos estrategias sobre todo el corpus
chunks_oraciones = aplicar_chunking(df, chunking_oraciones, oraciones_por_chunk=3, overlap_oraciones=1)
chunks_parrafos = aplicar_chunking(df, chunking_parrafos)

for nombre, lista in [("oraciones", chunks_oraciones), ("parrafos", chunks_parrafos)]:
    largos = [len(c["texto"]) for c in lista]
    print(f"{nombre}: {len(lista)} chunks, promedio {np.mean(largos):.0f} caracteres")


oraciones: 9251 chunks, promedio 182 caracteres
parrafos: 5018 chunks, promedio 270 caracteres


Como las reseñas son cortas, usamos un chunk por reseña (o por párrafo si existe). Esto reduce
duplicación causada por el solapamiento de oraciones y conserva mejor el contexto.

In [7]:
CHUNKS = chunks_parrafos
print(f"chunks finales: {len(CHUNKS)}")
print(CHUNKS[0])


chunks finales: 5018
{'texto': 'Tour fabuloso, bellísimo el bosque nuboso de Monteverde con el circuito de los puentes colgantes. Tras el almuerzo visitamos el ranario de la zona, donde pudimos observar gran variedad de sapos y ranitas con la ayuda del guía del lugar. Nuestro guía y conductor, Arturo, muy amable, atento y simpático. Recomendamos 100% esta actividad.', 'lugar': 'Excursión a los puentes colgantes de Monteverde', 'tipo_lugar': 'parque', 'calificacion': 5.0, 'polaridad': 'positiva', 'fuente': 'civitatis'}


### embeddings

un embedding es pasar el texto a un vector de numeros que representa el significado. la idea
es que textos parecidos en significado queden con vectores cerca entre si.

de aca para abajo hace falta internet la primera vez (descarga el modelo de hugging face,
pesa un rato)

In [8]:
from sentence_transformers import SentenceTransformer

NOMBRE_MODELO_EMBEDDINGS = 'paraphrase-multilingual-MiniLM-L12-v2'
modelo_embeddings = SentenceTransformer(NOMBRE_MODELO_EMBEDDINGS)
print("modelo cargado, dimension:", modelo_embeddings.get_sentence_embedding_dimension())

print("modelo listo para crear o cargar los embeddings")


modelo cargado, dimension: 384
modelo listo para crear o cargar los embeddings


### indice FAISS

FAISS es para no tener que comparar la pregunta contra todos los embeddings a mano, uno por
uno. usamos IndexFlatIP con los vectores normalizados, que da lo mismo que buscar por
similitud coseno 

In [9]:
import faiss

RUTA_CACHE = RUTA_PROYECTO / "data" / "embeddings_cache"
RUTA_INDICE = RUTA_CACHE / "indice.faiss"
RUTA_CHUNKS = RUTA_CACHE / "chunks.json"
RUTA_METADATA = RUTA_CACHE / "metadata.json"
hash_corpus = hashlib.sha256(RUTA_CORPUS.read_bytes()).hexdigest()
metadata_esperada = {"corpus_hash": hash_corpus, "modelo": NOMBRE_MODELO_EMBEDDINGS, "estrategia": "parrafos-v1"}

if RUTA_INDICE.exists() and RUTA_CHUNKS.exists() and RUTA_METADATA.exists() and json.loads(RUTA_METADATA.read_text(encoding="utf-8")) == metadata_esperada:
    indice = faiss.deserialize_index(np.frombuffer(RUTA_INDICE.read_bytes(), dtype=np.uint8))
    CHUNKS = json.loads(RUTA_CHUNKS.read_text(encoding="utf-8"))
    print(f"Índice cargado desde caché: {indice.ntotal} vectores")
else:
    textos = [c["texto"] for c in CHUNKS]
    embeddings = modelo_embeddings.encode(textos, show_progress_bar=True, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    indice = faiss.IndexFlatIP(embeddings.shape[1])
    indice.add(embeddings)
    RUTA_CACHE.mkdir(parents=True, exist_ok=True)
    # Evita el fallo de FAISS en Windows al escribir rutas con caracteres Unicode.
    RUTA_INDICE.write_bytes(faiss.serialize_index(indice).tobytes())
    RUTA_CHUNKS.write_text(json.dumps(CHUNKS, ensure_ascii=False), encoding="utf-8")
    RUTA_METADATA.write_text(json.dumps(metadata_esperada, ensure_ascii=False), encoding="utf-8")
    print(f"Índice creado y guardado: {indice.ntotal} vectores")


Índice cargado desde caché: 5018 vectores


### buscar chunks relevantes

esta es la parte de "retrieval": convierte la pregunta en embedding igual que a los chunks, y
le pide a FAISS los top_k mas parecidos

In [10]:
TIPOS_LUGAR = sorted({chunk["tipo_lugar"] for chunk in CHUNKS})

def detectar_tipos_lugar(pregunta):
    pregunta = pregunta.lower()
    return [tipo for tipo in TIPOS_LUGAR if tipo.split("_")[0] in pregunta]

def buscar_chunks_relevantes(pregunta, top_k=3, tipo_lugar=None, umbral=0.50):
    embedding_pregunta = modelo_embeddings.encode([pregunta], convert_to_numpy=True)
    faiss.normalize_L2(embedding_pregunta)

    candidatos = min(indice.ntotal, max(30, top_k * 10))
    scores, indices = indice.search(embedding_pregunta, candidatos)

    resultados, lugares_vistos = [], set()
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or score < umbral:
            continue
        chunk = CHUNKS[idx]
        if tipo_lugar and chunk["tipo_lugar"] != tipo_lugar:
            continue
        clave_lugar = chunk["lugar"].strip().lower()
        if clave_lugar in lugares_vistos:
            continue
        lugares_vistos.add(clave_lugar)
        resultados.append({
            "chunk": chunk["texto"],
            "lugar": chunk["lugar"],
            "tipo_lugar": chunk["tipo_lugar"],
            "calificacion": chunk["calificacion"],
            "polaridad": chunk["polaridad"],
            "fuente": chunk["fuente"],
            "score": float(score),
        })
        if len(resultados) == top_k:
            break
    return resultados


# prueba rapida a ver si trae algo coherente
for r in buscar_chunks_relevantes("hoteles con buena atencion al cliente"):
    print(f"[{r['score']:.3f}] ({r['tipo_lugar']} - {r['lugar']}) {r['chunk'][:120]}...")


[0.854] (hotel - Hotel Alajuela City) Excelente Hotel muy limpio y el personal muy amable lo recomiendo...
[0.847] (hotel - Hotel Park View) Muy buen hotel agradable...
[0.838] (hotel - Hotel Courtyard de Marriott • Alajuela) Perfecto lugar para estar Excelente Hotel...


### generar la respuesta

esta es la parte de "generation": le pasamos al LLM el contexto que encontramos mas  terminamos usando
Qwen2.5-1.5B-Instruct, que es multilingue de verdad y sigue instrucciones tipo chat mejor.


In [11]:
import ollama
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ServerError

# Generador local (Qwen vía Ollama). No descarga ni carga pesos en Python.
MODELO_OLLAMA = os.getenv("OLLAMA_MODEL", "qwen3:1.7b")

# Igual que en los notebooks del curso, la API es opcional: Qwen local funciona sin claves.
load_dotenv(RUTA_PROYECTO / ".env")
CLAVE_GEMINI = os.getenv("GEMINI_API_KEY")
MODELO_GEMINI = os.getenv("MODELO_GEMINI", "gemini-2.5-flash-lite")
cliente_gemini = genai.Client(api_key=CLAVE_GEMINI) if CLAVE_GEMINI else None

if cliente_gemini:
    print(f"Generadores listos: {MODELO_OLLAMA} local y Gemini opcional ({MODELO_GEMINI}).")
else:
    print(f"Generador listo: {MODELO_OLLAMA} local. Gemini está desactivado (no hay GEMINI_API_KEY).")


Generadores listos: qwen3:1.7b local y Gemini opcional (gemini-2.5-flash-lite).


In [12]:
INSTRUCCION_SISTEMA = "Responde en español usando SOLAMENTE las reseñas entregadas. Cada afirmación factual debe incluir la cita [n] de la reseña que la respalda. No inventes datos, no generalices y no infieras motivos. Si el contexto no permite responder, di exactamente: No tengo información suficiente en las reseñas recuperadas para responder esa pregunta."

def generar_respuesta_qwen(contexto, pregunta):
    if not contexto:
        return "No tengo información suficiente en las reseñas recuperadas para responder esa pregunta."

    mensajes = [
        {"role": "system", "content": INSTRUCCION_SISTEMA},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}
    ]
    salida = ollama.chat(model=MODELO_OLLAMA, messages=mensajes, options={"temperature": 0})
    return salida["message"]["content"]

def generar_respuesta_gemini(contexto, pregunta):
    if cliente_gemini is None:
        raise RuntimeError("Gemini es opcional y no está configurado. Agrega GEMINI_API_KEY a .env o usa modelo='qwen'.")
    if not contexto:
        return "No tengo información suficiente en las reseñas recuperadas para responder esa pregunta."
    prompt = f"{INSTRUCCION_SISTEMA}\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}"
    try:
        respuesta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0, max_output_tokens=220),
        )
        return respuesta.text
    except ServerError as error:
        if getattr(error, "code", None) == 503:
            return "Gemini no está disponible temporalmente (HTTP 503). Vuelve a ejecutar esta celda en unos minutos."
        raise

def generar_respuesta(contexto, pregunta, modelo="qwen"):
    if modelo == "qwen":
        return generar_respuesta_qwen(contexto, pregunta)
    if modelo == "gemini":
        return generar_respuesta_gemini(contexto, pregunta)
    raise ValueError("modelo debe ser 'qwen' o 'gemini'")

def generar_respuesta_sin_rag(pregunta, modelo="qwen"):
    """Baseline sin chunks del corpus; permite comparar el efecto real del RAG."""
    instruccion = ("Eres un asistente turístico de Costa Rica. Responde en español de forma breve. "
                   "No tienes acceso al corpus de reseñas; no cites ni inventes fuentes.")
    if modelo == "qwen":
        mensajes = [{"role": "system", "content": instruccion}, {"role": "user", "content": pregunta}]
        salida = ollama.chat(model=MODELO_OLLAMA, messages=mensajes, options={"temperature": 0})
        return salida["message"]["content"]
    if modelo == "gemini":
        if cliente_gemini is None:
            raise RuntimeError("Gemini no está configurado.")
        respuesta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI, contents=f"{instruccion}\n\nPregunta: {pregunta}",
            config=types.GenerateContentConfig(temperature=0, max_output_tokens=220),
        )
        return respuesta.text
    raise ValueError("modelo debe ser 'qwen' o 'gemini'")


### rag completo



In [13]:
def rag_completo(pregunta, top_k=3, tipos_lugar=None, umbral=0.50, modelo="qwen"):
    print(f"\nPREGUNTA: {pregunta}")

    tipos_lugar = tipos_lugar or detectar_tipos_lugar(pregunta)
    if len(tipos_lugar) > 1:
        por_tipo = max(1, top_k // len(tipos_lugar))
        resultados = [resultado for tipo in tipos_lugar for resultado in buscar_chunks_relevantes(pregunta, por_tipo, tipo, umbral)]
    else:
        tipo = tipos_lugar[0] if tipos_lugar else None
        resultados = buscar_chunks_relevantes(pregunta, top_k, tipo, umbral)
    for r in resultados:
        print(f"  - ({r['tipo_lugar']} - {r['lugar']}, score {r['score']:.3f}) {r['chunk'][:80]}...")

    contexto = "\n\n".join(
        f"[{i}] {r['tipo_lugar']} - {r['lugar']} ({r['calificacion']} estrellas, fuente: {r['fuente']}): {r['chunk']}"
        for i, r in enumerate(resultados, start=1)
    )

    respuesta = generar_respuesta(contexto, pregunta, modelo)
    print(f"\nRESPUESTA: {respuesta}")
    return respuesta

def comparar_con_y_sin_rag(pregunta, top_k=4, tipos_lugar=None, umbral=0.50, modelo="qwen"):
    """Ejecuta la misma pregunta con evidencia recuperada y sin ella."""
    respuesta_con_rag = rag_completo(pregunta, top_k, tipos_lugar, umbral, modelo)
    respuesta_sin_rag = generar_respuesta_sin_rag(pregunta, modelo)
    comparacion = {
        "pregunta": pregunta,
        "modelo": modelo,
        "respuesta_con_rag": respuesta_con_rag,
        "respuesta_sin_rag": respuesta_sin_rag,
    }
    print(f"\nSIN RAG: {respuesta_sin_rag}")
    return comparacion


### Pruebas y evaluación manual

Verifica que los lugares recuperados sean pertinentes y que cada afirmación de la respuesta tenga una cita `[n]`.

También se compara una misma pregunta con y sin RAG. La versión con RAG debe ser trazable a los fragmentos recuperados; la versión sin RAG sirve como baseline y no debe presentarse como evidencia del corpus.

In [14]:
CASOS_PRUEBA = [
    {"pregunta": "Que hoteles tienen buena atencion al cliente?", "tipos": ["hotel"]},
    {"pregunta": "Recomiendame un parque nacional con senderos bonitos", "tipos": ["parque"]},
    {"pregunta": "Que opinan de los museos en San Jose?", "tipos": ["museo"]},
    {"pregunta": "Que diferencia una reseña de un parque de una de un hotel?", "tipos": ["parque", "hotel"]},
    {"pregunta": "De que se quejan en los mercados artesanales?", "tipos": ["mercado"]},
]

for caso in CASOS_PRUEBA:
    rag_completo(caso["pregunta"], top_k=4, tipos_lugar=caso["tipos"], modelo="qwen")

# Comparación obligatoria: la misma pregunta con evidencia del RAG y sin evidencia.
caso = CASOS_PRUEBA[-1]
comparacion = comparar_con_y_sin_rag(caso["pregunta"], top_k=4, tipos_lugar=caso["tipos"], modelo="qwen")

RUTA_RESULTADO = RUTA_PROYECTO / "resultados" / "evaluacion_rag_sin_rag.json"
RUTA_RESULTADO.parent.mkdir(exist_ok=True)
RUTA_RESULTADO.write_text(json.dumps(comparacion, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nComparación guardada en: {RUTA_RESULTADO}")




PREGUNTA: Que hoteles tienen buena atencion al cliente?
  - (hotel - Hotel Alajuela City, score 0.825) Excelente Hotel muy limpio y el personal muy amable lo recomiendo...
  - (hotel - Hotel Courtyard de Marriott • Alajuela, score 0.806) Perfecto lugar para estar Excelente Hotel...
  - (hotel - Hotel Las Brumas, score 0.799) Muy buen hotel. buena atención, Buena instalación y muy cómodo. Recomendado...
  - (hotel - Hotel Marriott Hacienda Belén, score 0.798) Tuve una excelente experiencia en este hotel. Desde el primer momento la atenció...

RESPUESTA: Los hoteles con buena atención al cliente son los siguientes:  
- **Hotel Alajuela City** (revisión [1]): El personal es muy amable y el hotel es muy limpio.  
- **Hotel Courtyard de Marriott • Alajuela** (revisión [2]): El lugar es perfecto y el personal es súper amable y profesional.  
- **Hotel Las Brumas** (revisión [3]): La atención es buena, las instalaciones están bien cuidadas y el personal es amable.  
- **Hotel Marriott Hacien

### Prueba independiente con Gemini

Esta celda es opcional y se ejecuta manualmente solo cuando se desea comparar Gemini con el generador local. Requiere `GEMINI_API_KEY` en `.env`; no afecta el modo local de Qwen.

In [15]:
# Ejecuta para probar Gemini con el mismo RAG.
PREGUNTA_GEMINI = "¿Qué hoteles tienen buena atención al cliente?"

if cliente_gemini is None:
    print("Gemini no está configurado. Agrega GEMINI_API_KEY a .env y vuelve a ejecutar desde la sección de generación.")
else:
    print(f"Probando {MODELO_GEMINI} con RAG...")
    respuesta_gemini = rag_completo(
        PREGUNTA_GEMINI, top_k=4, tipos_lugar=["hotel"], modelo="gemini"
    )


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Probando gemini-2.5-flash-lite con RAG...

PREGUNTA: ¿Qué hoteles tienen buena atención al cliente?
  - (hotel - Hotel Alajuela City, score 0.802) Excelente Hotel muy limpio y el personal muy amable lo recomiendo...
  - (hotel - Hotel Marriott Hacienda Belén, score 0.789) Tuve una excelente experiencia en este hotel. Desde el primer momento la atenció...
  - (hotel - Hotel Las Brumas, score 0.779) Muy buen hotel. buena atención, Buena instalación y muy cómodo. Recomendado...
  - (hotel - Hotel Courtyard de Marriott • Alajuela, score 0.777) Perfecto lugar para estar Excelente Hotel...

RESPUESTA: El Hotel Alajuela City tiene personal muy amable [1]. El Hotel Marriott Hacienda Belén tiene una atención súper amable y profesional [2]. El Hotel Las Brumas tiene buena atención [3].
